In [1]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class VimEncoder(nn.Module):
    """Thin wrapper around mambapy's VMamba. The actual bidirectional
    scan (forward+backward combined per block) happens inside VMamba --
    this class just configures it and adds a final norm."""
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(
            d_model=d_model, n_layers=n_layers, d_state=d_state,
            bidirectional=True,   # forward + backward scan combined per block (Vim's core mechanism)
            divide_output=True,   # average the two directions instead of just adding them
            pscan=True,           # use the tested parallel scan, not a slow python loop
            use_cuda=False        
        )
        self.encoder = VMamba(config)
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [3]:
class ROIPatchEmbed3D(nn.Module):
    """Turns 6 ROI volumes into a token sequence for Vim.
    Each 64^3 ROI -> non-overlapping 8^3 patches -> one token per patch."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois = n_rois
        self.patch_size = patch_size
        self.grid_size = roi_size // patch_size          # patches per side (8)
        self.patches_per_roi = self.grid_size ** 3         # total patches per ROI (512)
        self.d_model = d_model

    
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)

        # Factorised positional info: which ROI + where within it (depth/height/width),
        # instead of one independent vector per patch position
        self.roi_embed = nn.Embedding(n_rois, d_model)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            # scale down from default N(0,1) init so position info doesn't
            # drown out the actual patch content early in training
            for emb in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)

        # precompute (depth, height, width) coordinate for every one of the 512 patch slots
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, rois):
        # rois: (B, 6, 1, 64, 64, 64)
        batch_size, n_rois = rois.shape[:2]
        x = rois.reshape(batch_size * n_rois, 1, rois.shape[-3], rois.shape[-2], rois.shape[-1])

        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)  # (B*6, 512, d_model)
        tokens = tokens.reshape(batch_size, n_rois, self.patches_per_roi, self.d_model)

        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, None, :, :] + self.roi_embed.weight[None, :, None, :]

        # identify entirely-empty (background) patches so they can be excluded
        occupancy = F.max_pool3d((x.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool().reshape(batch_size, n_rois, self.patches_per_roi)

        tokens = tokens.reshape(batch_size, -1, self.d_model)   # flatten to one sequence (3072 tokens)
        valid = valid.reshape(batch_size, -1)

        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)

        return tokens, valid


class VisionMambaBranch(nn.Module):
    """One modality's full pipeline: patch embed -> Vim -> pooling.
    Used twice (once per modality) in the multimodal model below."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        self.n_rois = n_rois
        self.patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.vim = VimEncoder(d_model, n_layers, d_state)

    def forward(self, rois):
        tokens, valid = self.patch_embed(rois)
        tokens = self.vim(tokens)   # (B, 3072, d_model), still per-token

        # masked mean pool: average only the valid (non-background) tokens
        w = valid.unsqueeze(-1).to(tokens.dtype)
        pooled = (tokens * w).sum(dim=1) / w.sum(dim=1).clamp_min(1.0)   # (B, d_model) -> classifier input

        # ALSO compute per-ROI pooled vectors 
        # saved for later explainability / ROI-contribution analysis)
        B = tokens.shape[0]
        tokens_by_roi = tokens.reshape(B, self.n_rois, -1, tokens.shape[-1])
        mask_by_roi = valid.reshape(B, self.n_rois, -1, 1).to(tokens.dtype)
        roi_embeddings = (tokens_by_roi * mask_by_roi).sum(dim=2) / mask_by_roi.sum(dim=2).clamp_min(1.0)

        return pooled, roi_embeddings


class VisionMambaModel(nn.Module):
    """Single-modality model -- same class used for MRI-only and PET-only,
    just point it at the MRI or PET cache directory."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, rois, return_roi_embeddings=False):
        pooled, roi_emb = self.branch(rois)
        logits = self.classifier(self.dropout(pooled))
        return (logits, roi_emb) if return_roi_embeddings else logits


class MultimodalVisionMambaModel(nn.Module):
    """Late fusion: two independent branches (separate weights), never
    share information until concatenation right before the classifier."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.mri_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.pet_branch = VisionMambaBranch(n_rois, roi_size, patch_size, d_model, n_layers, d_state)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)   # *2 because two branches concatenated

    def forward(self, mri_rois, pet_rois, return_roi_embeddings=False):
        mri_pooled, mri_emb = self.mri_branch(mri_rois)
        pet_pooled, pet_emb = self.pet_branch(pet_rois)
        logits = self.classifier(self.dropout(torch.cat([mri_pooled, pet_pooled], dim=1)))
        return (logits, mri_emb, pet_emb) if return_roi_embeddings else logits

In [4]:
COHORT_CSV     = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR       = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
if df["subject_id"].nunique() != len(df):
    raise ValueError("Split must be subject-level.")

sessions, labels = df["mri_session"].values, df["outcome_label"].values

# random_state=42 matches existing cached .npy augmentation files exactly --
# no need to regenerate anything with this split
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))   # PET cache is keyed by subject_id, not session

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [5]:
class ROIDataset(Dataset):
    """Single-modality dataset. Training set includes 3 augmented copies
    per subject (4x total); val/test use only the original scan."""
    def __init__(self, sessions, labels, cache_dir, is_mri=True, is_train=False):
        self.samples, self.cache_dir = [], cache_dir
        for session_id, label in zip(sessions, labels):
            key = session_id if is_mri else session_to_subject[session_id]  # MRI keys by session, PET by subject
            self.samples.append((key, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((key, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        key, label, version = self.samples[idx]
        rois = np.array(np.load(f"{self.cache_dir}/{key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(rois).unsqueeze(1), torch.tensor(label, dtype=torch.long), key


class MultimodalROIDataset(Dataset):
    """Pairs MRI and PET files for the SAME subject and SAME augmentation
    seed, so both modalities see a matched (though independently
    transformed) version of each training sample."""
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir, self.pet_cache_dir = mri_cache_dir, pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.array(np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        pet_rois = np.array(np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(mri_rois).unsqueeze(1), torch.from_numpy(pet_rois).unsqueeze(1), torch.tensor(label, dtype=torch.long), mri_key

In [6]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for rois, labels, _ in loader:  
        rois, labels = rois.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(rois), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for rois, labels, _ in loader:
            rois, labels = rois.to(device), labels.to(device)
            out = model(rois)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)   # sensitivity: catches converters correctly
    tnr = specificity_score(labels_all, preds_all)                # specificity: catches stable subjects correctly
    return total_loss / len(loader), acc, tpr, tnr

def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_rois, pet_rois, labels, _ in loader:
        mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(mri_rois, pet_rois), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for mri_rois, pet_rois, labels, _ in loader:
            mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
            out = model(mri_rois, pet_rois)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

In [7]:
def measure_inference_time(model, loader, device, is_multimodal, n_batches=20):
    """per-sample inference time"""
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            if is_multimodal:
                mri_rois, pet_rois, labels, _ = batch
                mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
                bs = mri_rois.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()   # ensure GPU actually finished, not just queued
                t0 = time.time()
                _ = model(mri_rois, pet_rois)
            else:
                rois, labels, _ = batch
                rois = rois.to(device)
                bs = rois.shape[0]
                if device.type == 'cuda': torch.cuda.synchronize()
                t0 = time.time()
                _ = model(rois)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.time() - t0) / bs)
    return np.mean(times), np.std(times)

def try_compute_flops(model, loader, device, is_multimodal):
    """thop counts standard Conv/Linear layers """
    try:
        model.eval()
        batch = next(iter(loader))
        with torch.no_grad():
            if is_multimodal:
                mri_rois, pet_rois, labels, _ = batch
                inputs = (mri_rois[:1].to(device), pet_rois[:1].to(device))   # just 1 sample needed
            else:
                rois, labels, _ = batch
                inputs = (rois[:1].to(device),)
            macs, _ = profile(model, inputs=inputs, verbose=False)
        return macs * 2   # 1 MAC = 2 FLOPs
    except Exception as e:
        print(f"  (FLOPs failed: {e})")
        return None

def run_one_seed(seed, model_class, train_loader, val_loader, test_loader, is_multimodal, save_prefix, max_epochs=101, patience=15):
    """Trains one model from a fixed random seed, early-stops on val loss,
    reloads the best checkpoint, then reports test performance + efficiency."""
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    model = model_class(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    train_fn = train_epoch_mm if is_multimodal else train_epoch
    eval_fn = evaluate_mm if is_multimodal else evaluate

    best_val_loss, no_improve, best_epoch, total_time = float("inf"), 0, 0, 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"

    print(f"\n--- Seed {seed} ---")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_fn(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = eval_fn(model, val_loader, criterion, device)
        scheduler.step(val_loss)   # reduces LR if val loss plateaus
        epoch_time = time.time() - t0
        total_time += epoch_time
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

        if val_loss < best_val_loss:   # save checkpoint ONLY when val loss improves (not accuracy)
            best_val_loss, best_epoch, no_improve = val_loss, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))   # reload best checkpoint, not final weights
    test_loss, test_acc, test_tpr, test_tnr = eval_fn(model, test_loader, criterion, device)   # test set touched exactly once
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_mean, inf_std = measure_inference_time(model, test_loader, device, is_multimodal)
    flops = try_compute_flops(model, test_loader, device, is_multimodal)

    print(f"\n  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_time/60:.1f}min | inf={inf_mean*1000:.2f}ms | {f'{flops/1e9:.2f}GFLOPs' if flops else 'N/A'}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch,
            "train_time_sec": total_time, "n_params": n_params, "inf_time_ms": inf_mean * 1000, "flops": flops}

In [8]:
# MRI ONLY
BATCH_SIZE = 4   
mri_train_loader = DataLoader(ROIDataset(X_train, y_train, MRI_CACHE_AUG, True, True), batch_size=BATCH_SIZE, shuffle=True)
mri_val_loader   = DataLoader(ROIDataset(X_val, y_val, MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)
mri_test_loader  = DataLoader(ROIDataset(X_test, y_test, MRI_CACHE_AUG, True, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== MRI-ONLY : 3-seed run ===")
mri_seed_results = [run_one_seed(s, VisionMambaModel, mri_train_loader, mri_val_loader, mri_test_loader,
                                  False, "vim_baseline_mri") for s in [1, 7, 123]]

=== MRI-ONLY : 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7052 |     0.6953 |   0.4762 |   0.9048 |   0.0476 |  10.2s
     2 |     0.6984 |     0.6924 |   0.5476 |   0.4762 |   0.6190 |   9.9s
     3 |     0.6981 |     0.6902 |   0.5238 |   0.2381 |   0.8095 |   9.9s
     4 |     0.6890 |     0.6891 |   0.4762 |   0.9524 |   0.0000 |   9.8s
     5 |     0.6905 |     0.6874 |   0.5476 |   0.8571 |   0.2381 |   9.9s
     6 |     0.6889 |     0.6857 |   0.5476 |   0.2857 |   0.8095 |   9.8s
     7 |     0.6883 |     0.6854 |   0.5476 |   0.6190 |   0.4762 |   9.9s
     8 |     0.6876 |     0.6841 |   0.5714 |   0.5238 |   0.6190 |   9.9s
     9 |     0.6874 |     0.6828 |   0.5714 |   0.3333 |   0.8095 |  10.0s
    10 |     0.6824 |     0.6812 |   0.5714 |   0.3333 |   0.8095 |  10.0s
    11 |     0.6866 |     0.6815 |   0.5714 |   0.1905 |  

In [9]:
# PET ONLY
pet_train_loader = DataLoader(ROIDataset(X_train, y_train, PET_CACHE_AUG, False, True), batch_size=BATCH_SIZE, shuffle=True)
pet_val_loader   = DataLoader(ROIDataset(X_val, y_val, PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)
pet_test_loader  = DataLoader(ROIDataset(X_test, y_test, PET_CACHE_AUG, False, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== PET-ONLY : 3-seed run ===")
pet_seed_results = [run_one_seed(s, VisionMambaModel, pet_train_loader, pet_val_loader, pet_test_loader,
                                  False, "vim_baseline_pet") for s in [1, 7, 123]]

=== PET-ONLY : 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.7056 |     0.6875 |   0.6190 |   0.4762 |   0.7619 |  50.9s
     2 |     0.6959 |     0.6852 |   0.5476 |   0.1905 |   0.9048 |  10.0s
     3 |     0.6946 |     0.6845 |   0.5714 |   0.2381 |   0.9048 |   9.9s
     4 |     0.6885 |     0.6838 |   0.5714 |   0.5714 |   0.5714 |   9.9s
     5 |     0.6891 |     0.6835 |   0.5714 |   0.4762 |   0.6667 |   9.9s
     6 |     0.6863 |     0.6830 |   0.6190 |   0.3333 |   0.9048 |   9.9s
     7 |     0.6867 |     0.6816 |   0.5714 |   0.4286 |   0.7143 |   9.9s
     8 |     0.6837 |     0.6817 |   0.5476 |   0.4762 |   0.6190 |   9.9s
     9 |     0.6838 |     0.6811 |   0.5952 |   0.4286 |   0.7619 |   9.9s
    10 |     0.6803 |     0.6807 |   0.5952 |   0.3810 |   0.8095 |   9.9s
    11 |     0.6828 |     0.6800 |   0.6190 |   0.3333 |  

In [10]:
# MIXED MODALITY (MRI & PET)
mm_train_loader = DataLoader(MultimodalROIDataset(X_train, y_train, MRI_CACHE_AUG, PET_CACHE_AUG, True), batch_size=4, shuffle=True)
mm_val_loader   = DataLoader(MultimodalROIDataset(X_val, y_val, MRI_CACHE_AUG, PET_CACHE_AUG, False), batch_size=4, shuffle=False)
mm_test_loader  = DataLoader(MultimodalROIDataset(X_test, y_test, MRI_CACHE_AUG, PET_CACHE_AUG, False), batch_size=4, shuffle=False)

print("=== MULTIMODAL : 3-seed run ===")
mm_seed_results = [run_one_seed(s, MultimodalVisionMambaModel, mm_train_loader, mm_val_loader, mm_test_loader,
                                 True, "vim_baseline_mm") for s in [1, 7, 123]]

=== MULTIMODAL : 3-seed run ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.6963 |     0.6900 |   0.5000 |   1.0000 |   0.0000 |  19.7s
     2 |     0.6997 |     0.6953 |   0.5000 |   1.0000 |   0.0000 |  19.7s
     3 |     0.6945 |     0.6887 |   0.5000 |   1.0000 |   0.0000 |  19.7s
     4 |     0.6899 |     0.6836 |   0.5952 |   0.2381 |   0.9524 |  19.7s
     5 |     0.6849 |     0.6823 |   0.5238 |   0.3333 |   0.7143 |  19.7s
     6 |     0.6833 |     0.6822 |   0.5714 |   0.1905 |   0.9524 |  19.7s
     7 |     0.6895 |     0.6813 |   0.5238 |   0.6667 |   0.3810 |  19.7s
     8 |     0.6814 |     0.6807 |   0.5476 |   0.7619 |   0.3333 |  19.7s
     9 |     0.6855 |     0.6782 |   0.5476 |   0.6190 |   0.4762 |  19.6s
    10 |     0.6744 |     0.6760 |   0.5238 |   0.4286 |   0.6190 |  19.6s
    11 |     0.6758 |     0.6744 |   0.5952 |   0.3810 |

In [11]:
def summarize(results, name):
    accs, tprs, tnrs = [r["acc"] for r in results], [r["tpr"] for r in results], [r["tnr"] for r in results]
    times, infs = [r["train_time_sec"] for r in results], [r["inf_time_ms"] for r in results]
    flops_vals = [r["flops"] for r in results if r["flops"]]
    print(f"\n{name}: Acc={np.mean(accs)*100:.1f}±{np.std(accs,ddof=1)*100:.1f}% | "
          f"TPR={np.mean(tprs)*100:.1f}±{np.std(tprs,ddof=1)*100:.1f}% | "
          f"TNR={np.mean(tnrs)*100:.1f}±{np.std(tnrs,ddof=1)*100:.1f}% | "
          f"Params={results[0]['n_params']:,} | Train={np.mean(times)/60:.1f}m | Inf={np.mean(infs):.2f}ms")

print(f"{'MNA-net (baseline)':<20}: Acc=82.9% | TPR=85.7% | TNR=80.0%")   # Vo et al., single-seed reference
summarize(mri_seed_results, "MRI-only (no stats)")
summarize(pet_seed_results, "PET-only (no stats)")
summarize(mm_seed_results, "Multimodal (no stats)")

MNA-net (baseline)  : Acc=82.9% | TPR=85.7% | TNR=80.0%

MRI-only (no stats): Acc=68.3±2.7% | TPR=63.5±7.3% | TNR=73.0±2.7% | Params=44,962 | Train=9.5m | Inf=5.10ms

PET-only (no stats): Acc=62.7±5.0% | TPR=68.3±12.0% | TNR=57.1±4.8% | Params=44,962 | Train=10.8m | Inf=6.58ms

Multimodal (no stats): Acc=65.1±1.4% | TPR=58.7±5.5% | TNR=71.4±4.8% | Params=89,922 | Train=24.0m | Inf=10.08ms


In [12]:
def evaluate_prob_fusion(mri_model, pet_model, mm_test_loader, device):
    mri_model.eval()
    pet_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for mri_rois, pet_rois, labels, _ in mm_test_loader:
            mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
            mri_probs = torch.softmax(mri_model(mri_rois), dim=1)
            pet_probs = torch.softmax(pet_model(pet_rois), dim=1)
            fused_probs = 0.5 * mri_probs + 0.5 * pet_probs
            preds = fused_probs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    tpr = recall_score(all_labels, all_preds, zero_division=0)
    tnr = specificity_score(all_labels, all_preds)
    return acc, tpr, tnr


fusion_results = []
for seed in [1, 7, 123]:
    mri_model_fusion = VisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    mri_model_fusion.load_state_dict(torch.load(f"{CKPT_DIR}/vim_baseline_mri_seed{seed}.pt", weights_only=True))

    pet_model_fusion = VisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    pet_model_fusion.load_state_dict(torch.load(f"{CKPT_DIR}/vim_baseline_pet_seed{seed}.pt", weights_only=True))

    acc, tpr, tnr = evaluate_prob_fusion(mri_model_fusion, pet_model_fusion, mm_test_loader, device)
    print(f"Seed {seed}: Acc={acc*100:.1f}% | TPR={tpr*100:.1f}% | TNR={tnr*100:.1f}%")
    fusion_results.append({"acc": acc, "tpr": tpr, "tnr": tnr})

accs = [r["acc"] for r in fusion_results]
tprs = [r["tpr"] for r in fusion_results]
tnrs = [r["tnr"] for r in fusion_results]
print(f"\nProbability fusion (no stats): Acc={np.mean(accs)*100:.1f}±{np.std(accs,ddof=1)*100:.1f}% | "
      f"TPR={np.mean(tprs)*100:.1f}±{np.std(tprs,ddof=1)*100:.1f}% | "
      f"TNR={np.mean(tnrs)*100:.1f}±{np.std(tnrs,ddof=1)*100:.1f}%")
print(f"Compare to joint-trained multimodal (no stats): Acc=65.1±1.4% | TPR=58.7±5.5% | TNR=71.4±4.8%")

Seed 1: Acc=71.4% | TPR=71.4% | TNR=71.4%
Seed 7: Acc=66.7% | TPR=76.2% | TNR=57.1%
Seed 123: Acc=64.3% | TPR=66.7% | TNR=61.9%

Probability fusion (no stats): Acc=67.5±3.6% | TPR=71.4±4.8% | TNR=63.5±7.3%
Compare to joint-trained multimodal (no stats): Acc=65.1±1.4% | TPR=58.7±5.5% | TNR=71.4±4.8%


In [14]:
from sklearn.linear_model import LogisticRegression

def get_probs_and_labels(model, loader, device):
    model.eval()
    probs_all, labels_all = [], []
    with torch.no_grad():
        for rois, labels, _ in loader:
            rois = rois.to(device)
            probs = torch.softmax(model(rois), dim=1)[:, 1]  # P(converter)
            probs_all.extend(probs.cpu().numpy())
            labels_all.extend(labels.numpy())
    return np.array(probs_all), np.array(labels_all)


def get_probs_only_multimodal(mri_model, pet_model, mm_loader, device):
    """Gets matched MRI+PET probabilities per subject from a multimodal loader."""
    mri_model.eval(); pet_model.eval()
    mri_probs_all, pet_probs_all, labels_all = [], [], []
    with torch.no_grad():
        for mri_rois, pet_rois, labels, _ in mm_loader:
            mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
            mri_p = torch.softmax(mri_model(mri_rois), dim=1)[:, 1]
            pet_p = torch.softmax(pet_model(pet_rois), dim=1)[:, 1]
            mri_probs_all.extend(mri_p.cpu().numpy())
            pet_probs_all.extend(pet_p.cpu().numpy())
            labels_all.extend(labels.numpy())
    return np.array(mri_probs_all), np.array(pet_probs_all), np.array(labels_all)


stacking_results = []
for seed in [1, 7, 123]:
    mri_model_fusion = VisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    mri_model_fusion.load_state_dict(torch.load(f"{CKPT_DIR}/vim_baseline_mri_seed{seed}.pt", weights_only=True))

    pet_model_fusion = VisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    pet_model_fusion.load_state_dict(torch.load(f"{CKPT_DIR}/vim_baseline_pet_seed{seed}.pt", weights_only=True))

    # Train the meta-learner on VALIDATION probabilities only
    mri_val_p, pet_val_p, val_labels = get_probs_only_multimodal(mri_model_fusion, pet_model_fusion, mm_val_loader, device)
    X_val_meta = np.stack([mri_val_p, pet_val_p], axis=1)  # (n_val, 2)

    meta_learner = LogisticRegression()
    meta_learner.fit(X_val_meta, val_labels)

    # Apply once to TEST probabilities
    mri_test_p, pet_test_p, test_labels = get_probs_only_multimodal(mri_model_fusion, pet_model_fusion, mm_test_loader, device)
    X_test_meta = np.stack([mri_test_p, pet_test_p], axis=1)
    preds = meta_learner.predict(X_test_meta)

    acc = np.mean(preds == test_labels)
    tpr = recall_score(test_labels, preds, zero_division=0)
    tnr = specificity_score(test_labels, preds)
    print(f"Seed {seed}: learned weights [MRI={meta_learner.coef_[0][0]:.2f}, PET={meta_learner.coef_[0][1]:.2f}] | "
          f"Acc={acc*100:.1f}% | TPR={tpr*100:.1f}% | TNR={tnr*100:.1f}%")
    stacking_results.append({"acc": acc, "tpr": tpr, "tnr": tnr})

accs = [r["acc"] for r in stacking_results]
tprs = [r["tpr"] for r in stacking_results]
tnrs = [r["tnr"] for r in stacking_results]
print(f"\nStacked fusion (learned meta-classifier): Acc={np.mean(accs)*100:.1f}±{np.std(accs,ddof=1)*100:.1f}% | "
      f"TPR={np.mean(tprs)*100:.1f}±{np.std(tprs,ddof=1)*100:.1f}% | "
      f"TNR={np.mean(tnrs)*100:.1f}±{np.std(tnrs,ddof=1)*100:.1f}%")
print(f"Compare to flat 50/50 fusion: Acc=67.5±3.6% | TPR=71.4±4.8% | TNR=63.5±7.3%")

Seed 1: learned weights [MRI=1.18, PET=1.06] | Acc=71.4% | TPR=71.4% | TNR=71.4%
Seed 7: learned weights [MRI=0.39, PET=1.39] | Acc=64.3% | TPR=66.7% | TNR=61.9%
Seed 123: learned weights [MRI=1.06, PET=1.21] | Acc=59.5% | TPR=61.9% | TNR=57.1%

Stacked fusion (learned meta-classifier): Acc=65.1±6.0% | TPR=66.7±4.8% | TNR=63.5±7.3%
Compare to flat 50/50 fusion: Acc=67.5±3.6% | TPR=71.4±4.8% | TNR=63.5±7.3%
